In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("Imports successful.")
print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")

In [ ]:
column_names = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
    'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target'
]

# UCI file uses '?' for missing values — tell pandas to treat them as NaN
df = pd.read_csv(
    '../data/raw/heart_disease.csv',
    names=column_names,
    na_values='?'
)

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
print("=" * 60)
print("DATA TYPES AND NON-NULL COUNTS")
print("=" * 60)
df.info()

print("\n" + "=" * 60)
print("SUMMARY STATISTICS")
print("=" * 60)
df.describe()

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct
})
missing_summary = missing_summary[missing_summary['missing_count'] > 0]

if len(missing_summary) == 0:
    print("No missing values.")
else:
    print("Columns with missing values:")
    print(missing_summary)

In [ ]:
print("Target value counts:")
print(df['target'].value_counts().sort_index())
print("\nTarget percentages:")
print((df['target'].value_counts(normalize=True).sort_index() * 100).round(2))

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['target'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Target Distribution (Multi-class: 0 = no disease, 1-4 = severity)')
axes[0].set_xlabel('Target')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Binarize: 0 = no disease, 1 = any disease
df['target_binary'] = (df['target'] > 0).astype(int)
df['target_binary'].value_counts().sort_index().plot(kind='bar', ax=axes[1], color=['steelblue', 'coral'])
axes[1].set_title('Target Distribution (Binary: 0 = healthy, 1 = disease)')
axes[1].set_xlabel('Target (binary)')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

print("\nBinary target percentages:")
print((df['target_binary'].value_counts(normalize=True).sort_index() * 100).round(2))

In [ ]:
numerical_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    axes[i].hist(df[col].dropna(), bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    axes[i].axvline(df[col].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df[col].mean():.1f}')
    axes[i].axvline(df[col].median(), color='green', linestyle='--', linewidth=2, label=f'Median: {df[col].median():.1f}')
    axes[i].set_title(f'Distribution of {col}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')
    axes[i].legend()

# Hide the empty 6th subplot
axes[5].axis('off')

plt.suptitle('Numerical Feature Distributions', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Drop the multi-class target; keep target_binary as our outcome
df_corr = df.drop(columns=['target']).copy()

correlation = df_corr.corr()

plt.figure(figsize=(12, 10))
sns.heatmap(
    correlation,
    annot=True,           # show numbers in cells
    fmt='.2f',            # 2 decimal places
    cmap='coolwarm',      # blue = negative, red = positive
    center=0,             # 0 = white
    square=True,
    cbar_kws={'shrink': 0.8},
    linewidths=0.5
)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
numerical_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    df.boxplot(column=col, by='target_binary', ax=axes[i], grid=False)
    axes[i].set_title(f'{col} by Disease Status', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Target (0 = healthy, 1 = disease)')
    axes[i].set_ylabel(col)
    axes[i].get_figure().suptitle('')  # remove default suptitle

axes[5].axis('off')

plt.suptitle('Numerical Features: Healthy vs. Disease Patients', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
categorical_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal']

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    crosstab = pd.crosstab(df[col], df['target_binary'], normalize='index') * 100
    crosstab.plot(kind='bar', stacked=True, ax=axes[i], color=['steelblue', 'coral'])
    axes[i].set_title(f'{col} vs. Disease Status (%)', fontsize=11, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Percentage')
    axes[i].legend(['Healthy', 'Disease'], loc='upper right', fontsize=8)
    axes[i].tick_params(axis='x', rotation=0)

axes[7].axis('off')

plt.tight_layout()
plt.show()

## EDA Summary

**Dataset:** UCI Heart Disease (Cleveland subset). 303 patients, 14 features.

**Target:** Reframed from 5-class severity (0-4) to binary (0 = healthy, 1 = disease).
Binary distribution: 54% healthy, 46% disease — roughly balanced.

**Missing values:** 6 total (1.32% in `ca`, 0.66% in `thal`). Will use median imputation.

**Demographic skew:** 68% male / 32% female. Model performance on female patients
should be evaluated separately before clinical use.

**Outlier:** One patient with cholesterol = 564 (extreme). Verified clinically plausible
(rare familial hypercholesterolemia). Retained in dataset; tree models robust to single outliers.

### Strongest predictors of disease (from EDA):
| Feature | Correlation | Pattern |
|---------|-------------|---------|
| thal    | +0.53 | Abnormal thalassemia (types 6, 7) → 70-80% disease rate |
| ca      | +0.46 | More diseased vessels → more disease |
| exang   | +0.43 | Exercise-induced angina → 75% disease |
| thalach | -0.42 | Lower max heart rate → more disease |
| oldpeak | +0.42 | Higher ST depression → more disease |
| cp      | +0.41 | Asymptomatic chest pain (type 4) → 75% disease (counterintuitive) |

### Weakest predictors:
- `chol` (+0.09), `fbs` (+0.03), `trestbps` (+0.15) — surprisingly weak in this cohort.

### Modeling implications:
- Tree-based models likely to outperform linear models due to skew and non-linear relationships.
- Mild multicollinearity (`slope` ↔ `oldpeak` = 0.58) — manageable, regularization helps.
- Will use SMOTE despite mild imbalance to demonstrate technique and lift minority recall.